# **Full Pipeline**

In [1]:
from pathlib import Path
import sys, os, platform

CWD  = Path.cwd().resolve()
ROOT = CWD if (CWD / "src").exists() else CWD.parent
if str(ROOT) not in sys.path: sys.path.append(str(ROOT))

In [2]:
DOC_ID = "New_York_State_Workers_Compensation_Medical_Fee"   # e.g., "title17"
PDF = ROOT / "data" / "raw" / f"{DOC_ID}.pdf"
RUN_DIR = ROOT / "data" / "runs" / DOC_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
MD_DIR     = RUN_DIR / "md"
CHUNKS_DIR = RUN_DIR / "chunks"
INDEX_DIR  = RUN_DIR / "index"
SFT_DIR    = RUN_DIR / "sft"
PAIRS_DIR  = RUN_DIR / "pairs"

print(f"[run] DOC_ID={DOC_ID}")
print(f"[pdf] {PDF}")
print(f"[out] {RUN_DIR}")

[run] DOC_ID=New_York_State_Workers_Compensation_Medical_Fee
[pdf] D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\raw\New_York_State_Workers_Compensation_Medical_Fee.pdf
[out] D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee


### **1) PDF to MD**

In [11]:
from src.ingest.pdf_to_markdown import convert_pdf_to_markdown
paths = convert_pdf_to_markdown(PDF, MD_DIR, ocr=True)
print("markdown pages:", paths)

markdown pages: {'markdown': WindowsPath('D:/IIT BBS/Job Resources/Business Optima/pdf-agent/data/runs/New_York_State_Workers_Compensation_Medical_Fee/md/New_York_State_Workers_Compensation_Medical_Fee.md'), 'toc_json': WindowsPath('D:/IIT BBS/Job Resources/Business Optima/pdf-agent/data/runs/New_York_State_Workers_Compensation_Medical_Fee/md/New_York_State_Workers_Compensation_Medical_Fee.toc.json'), 'pages_jsonl': WindowsPath('D:/IIT BBS/Job Resources/Business Optima/pdf-agent/data/runs/New_York_State_Workers_Compensation_Medical_Fee/md/New_York_State_Workers_Compensation_Medical_Fee.pages.jsonl'), 'images_dir': None}


### **2) Chunking**

In [12]:
from src.ingest.md_to_chunks import md_to_chunks
CHUNKS = CHUNKS_DIR / "chunks.jsonl"
n = md_to_chunks(
    MD_DIR / f"{DOC_ID}.md", CHUNKS,
    pages_jsonl=MD_DIR / f"{DOC_ID}.pages.jsonl",
    max_chars=1600, overlap=400, drop_gibberish=True, drop_toc=False
)
print("chunks:", n)

chunks: 1494


### **3) Index Building**

In [13]:
from src.ingest.build_index import build_index
collection_name = build_index(
    CHUNKS, persist=INDEX_DIR,
    embed_model="BAAI/bge-base-en-v1.5",
    batch_size=64, bge_use_prompt=True, reset=True
)
print("collection:", collection_name)

Indexed 64/1494
Indexed 128/1494
Indexed 192/1494
Indexed 256/1494
Indexed 320/1494
Indexed 384/1494
Indexed 448/1494
Indexed 512/1494
Indexed 576/1494
Indexed 640/1494
Indexed 704/1494
Indexed 768/1494
Indexed 832/1494
Indexed 896/1494
Indexed 960/1494
Indexed 1024/1494
Indexed 1088/1494
Indexed 1152/1494
Indexed 1216/1494
Indexed 1280/1494
Indexed 1344/1494
Indexed 1408/1494
Indexed 1472/1494
Indexed 1494/1494
[OK] Chroma collection 'New_York_State_Workers_Compensation_Medical_Fee' built at D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\index
[INFO] Embed model: BAAI/bge-base-en-v1.5 | BGE passage prompt: True
collection: New_York_State_Workers_Compensation_Medical_Fee


### **4) SFT Data Gen**

In [4]:
from src.ingest.make_qa_and_summaries import make_data

CHUNKS = RUN_DIR / "chunks" / "chunks.jsonl"

summaries_path, qa_path = make_data(
    chunks_path=CHUNKS,
    out_dir=SFT_DIR,
    model="llama3:instruct", # llama3.1:8b-instruct-q8_0
    max_sections=5, min_tokens_per_section=400,
    qa_per_section=3
)

[1/5] Section: Assembly Instructions > Effective April 1, 2019 | chars=15683 | pages=(1, 35)
[2/5] Section: Assembly Instructions > NEW CPT CODES | chars=9057 | pages=(10, 83)
[3/5] Section: Assembly Instructions > Changed Values | chars=8385 | pages=(10, 83)
[4/5] Section: Assembly Instructions > 52 Reduced Services | chars=7941 | pages=(18, 24)
[5/5] Section: Assembly Instructions > 1A. NYS Medical Treatment Guidelines | chars=7551 | pages=(13, 77)
[OK] Wrote 5 summaries → D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sft\summaries.jsonl
[OK] Wrote 5 Q/A → D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sft\qa.jsonl


### **5) Sanitize Summaries + QA (v3b polish)**

In [5]:
from pathlib import Path
import json, re, itertools

# Inputs from SFT gen
summ_in  = SFT_DIR / "summaries.jsonl"
qa_in    = SFT_DIR / "qa.jsonl"

# Outputs after cleaning
summ_out = SFT_DIR / "summaries.v3b.jsonl"
qa_out   = SFT_DIR / "qa.v3b.jsonl"

_SENT_SPLIT = re.compile(r'(?<=[.!?])\s+')
_STOP_LAST  = {"the","and","or","of","to","for","with","in","on","at","by"}

def _strip_bullet_prefix(s: str) -> str:
    s = re.sub(r'^\s*[-•·]+[\s\u00A0]+', '', s.strip())
    s = s.lstrip('"\''"“”‘’").strip()
    return s

def _normalize_punct(s: str) -> str:
    s = s.replace('..', '.').replace(',.', '.').replace('",', ',').replace('".', '.')
    s = re.sub(r'\s{2,}', ' ', s)
    return s.strip()

def _end_at_sentence(s: str) -> str:
    s = re.sub(r'\s*\[[^\]]*$', '', s.strip())   # strip dangling citations
    if not s.endswith(('.', '!', '?')):
        s += '.'
    parts = [p.strip() for p in _SENT_SPLIT.split(s) if p.strip()]
    if parts and len(parts[-1]) < 8 and len(parts) > 1:
        s = " ".join(parts[:-1]).strip()
        if not s.endswith(('.', '!', '?')):
            s += '.'
    return s

def _bullets_from_text(raw: str, max_items: int = 12, min_len: int = 20):
    sents = [t.strip() for t in _SENT_SPLIT.split(raw or "") if t.strip()]
    out = []
    for t in sents:
        t = _strip_bullet_prefix(_normalize_punct(t))
        t = _end_at_sentence(t)
        if len(t) >= min_len and not t.endswith((':',';')):
            out.append(f"- {t}")
        if len(out) >= max_items: break
    return out

def clean_summaries_v3b(path_in: Path, path_out: Path,
                        med_items=6, long_items=12, min_len_short=60):
    kept = dropped = 0
    out = []
    for line in open(path_in, "r", encoding="utf-8"):
        row = json.loads(line)
        short = _normalize_punct(_strip_bullet_prefix(str(row.get("short",""))))
        short = _end_at_sentence(short)

        med_bul = _bullets_from_text(str(row.get("medium","")), max_items=med_items)
        if len(med_bul) < max(3, med_items//2):
            med_bul = _bullets_from_text(str(row.get("long","")) or short, max_items=med_items)

        long_bul = _bullets_from_text(str(row.get("long","")), max_items=long_items)
        if not long_bul: long_bul = med_bul

        row["short"]  = short
        row["medium"] = "\n".join(med_bul)
        row["long"]   = "\n".join(long_bul)

        if len(row["short"]) < min_len_short:
            dropped += 1
            continue
        out.append(row); kept += 1

    with open(path_out, "w", encoding="utf-8") as f:
        for r in out: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"[OK] summaries v3b → {path_out} | kept={kept} dropped={dropped}")

def _trim_if_ends_on_stopword(ans: str) -> str:
    toks = re.findall(r"[A-Za-z]+", ans)
    if toks and toks[-1].lower() in _STOP_LAST:
        m = re.search(r'(?s)(.*[.!?])\s+[^.!?]*$', ans)
        if m: return m.group(1).strip()
    return ans

def clean_qa_v3b(path_in: Path, path_out: Path, min_len=60):
    kept = dropped = 0
    out = []
    for line in open(path_in, "r", encoding="utf-8"):
        row = json.loads(line)
        ans = _normalize_punct(str(row.get("answer","")).strip())
        ans = _end_at_sentence(ans)
        ans = _trim_if_ends_on_stopword(ans)
        if len(ans) < min_len:
            dropped += 1
            continue
        row["answer"] = ans
        out.append(row); kept += 1

    with open(path_out, "w", encoding="utf-8") as f:
        for r in out: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"[OK] qa v3b → {path_out} | kept={kept} dropped={dropped}")

# --- Run sanitizers ---
clean_summaries_v3b(summ_in, summ_out)
clean_qa_v3b(qa_in, qa_out)

# Peek
print("\nSummaries v3b sample:")
for ln in itertools.islice(open(summ_out, "r", encoding="utf-8"), 2):
    print(json.loads(ln))
print("\nQA v3b sample:")
for ln in itertools.islice(open(qa_out, "r", encoding="utf-8"), 2):
    print(json.loads(ln))

[OK] summaries v3b → D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sft\summaries.v3b.jsonl | kept=5 dropped=0
[OK] qa v3b → D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sft\qa.v3b.jsonl | kept=3 dropped=2

Summaries v3b sample:
{'section': 'Assembly Instructions > Effective April 1, 2019', 'pages': [1, 35], 'short': '{ "short": "The passage discusses various medical procedures including debridement, paring/cutting, and biopsy. Debridement involves removing foreign material at an open fracture or dislocation site, while paring/cutting removes benign hyperkeratotic lesions. Biopsy excises skin, subcutaneous tissue, and/or mucous.', 'medium': '- { "short": "The passage discusses various medical procedures including debridement, paring/cutting, and biopsy.\n- Debridement involves removing foreign material at an open fracture or dislocation site, while paring/cuttin

### **6) Build Train/Dev/Test Splits**

In [6]:
from src.train.sft_build import build_sft

train_p, dev_p, test_p = build_sft(
    summaries_path=summ_out,
    qa_path=qa_out,
    out_dir=SFT_DIR,
    train_frac=0.85, dev_frac=0.075, test_frac=0.075
)
print("Splits ready:\n", train_p, dev_p, test_p)

[OK] SFT examples: total=18 | train=15 dev=1 test=2
Splits ready:
 D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sft\train.jsonl D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sft\dev.jsonl D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sft\test.jsonl


### **7) LoRA/PEFT Training**

In [7]:
from pathlib import Path
import sys, os, platform

OUT = ROOT / "outputs" / "lora_hf" / DOC_ID
OUT.mkdir(parents=True, exist_ok=True)

LOCAL_QWEN = ROOT / "models" / "Qwen2.5-1.5B-Instruct"
MODEL_ID = str(LOCAL_QWEN)

os.environ["TRANSFORMERS_OFFLINE"] = "1"
LOCAL_FILES_ONLY = True

print("Model:", MODEL_ID, "| OUT:", OUT)

Model: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\models\Qwen2.5-1.5B-Instruct | OUT: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\outputs\lora_hf\New_York_State_Workers_Compensation_Medical_Fee


In [8]:
# Alpaca format conversion
from src.train.sft_to_alpaca import to_alpaca
ALPACA = SFT_DIR / "alpaca.train.jsonl"
if not ALPACA.exists():
    to_alpaca(train_p, ALPACA)

print("Alpaca file:", ALPACA)

[OK] wrote 15 alpaca examples → D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sft\alpaca.train.jsonl
Alpaca file: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sft\alpaca.train.jsonl


In [10]:
# Hyperparams
MAX_STEPS    = 150
BATCH_SIZE   = 1
GRAD_ACCUM   = 4
LR           = 2e-4
MAX_SEQ_LEN  = 512
NUM_THREADS  = min(8, os.cpu_count() or 8)
LORA_R       = 8
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05
SEED         = 7

In [11]:
# Launch training
import subprocess, shlex
args = [
    sys.executable, str(ROOT / "src/train/cpu_lora_hf.py"),
    "--model_id", MODEL_ID,
    "--train_jsonl", str(ALPACA),
    "--out_dir", str(OUT),
    "--max_steps", str(MAX_STEPS),
    "--batch_size", str(BATCH_SIZE),
    "--grad_accum", str(GRAD_ACCUM),
    "--lr", str(LR),
    "--max_seq_len", str(MAX_SEQ_LEN),
    "--num_threads", str(NUM_THREADS),
    "--lora_r", str(LORA_R),
    "--lora_alpha", str(LORA_ALPHA),
    "--lora_dropout", str(LORA_DROPOUT),
    "--seed", str(SEED),
    "--local_files_only",
    "--trust_remote_code",
]
print("Launching:", " ".join(map(shlex.quote, args)))
proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout: print(line, end="")
code = proc.wait()
print("\n[proc exit code]", code)

Launching: 'd:\Anaconda\envs\pdf-agent-2\python.exe' 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\src\train\cpu_lora_hf.py' --model_id 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\models\Qwen2.5-1.5B-Instruct' --train_jsonl 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sft\alpaca.train.jsonl' --out_dir 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\outputs\lora_hf\New_York_State_Workers_Compensation_Medical_Fee' --max_steps 150 --batch_size 1 --grad_accum 4 --lr 0.0002 --max_seq_len 512 --num_threads 8 --lora_r 8 --lora_alpha 16 --lora_dropout 0.05 --seed 7 --local_files_only --trust_remote_code

Generating train split: 0 examples [00:00, ? examples/s]
Generating train split: 15 examples [00:00, 78.55 examples/s]
Generating train split: 15 examples [00:00, 77.73 examples/s]

Map: 100%|██████████| 15/15 [00:00<00:00, 170.07 examples/s]
trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%

### **9) Adapter inspection**

In [12]:
from pathlib import Path

adapter_dir = OUT / "adapter" 
print("Adapter dir:", adapter_dir, "| exists:", adapter_dir.exists())
if adapter_dir.exists():
    items = sorted(p.name for p in adapter_dir.iterdir())
    for name in items:
        print(" -", name)
else:
    print("[WARN] Adapter folder not found. Did training finish successfully?")

Adapter dir: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\outputs\lora_hf\New_York_State_Workers_Compensation_Medical_Fee\adapter | exists: True
 - README.md
 - adapter_config.json
 - adapter_model.safetensors
 - added_tokens.json
 - chat_template.jinja
 - merges.txt
 - special_tokens_map.json
 - tokenizer.json
 - tokenizer_config.json
 - vocab.json


### **10) Closed book Inference with the adapter (base + PEFT overlay)**

In [13]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_path  = MODEL_ID
adapter_dir = OUT / "adapter" # the LoRA adapter

# Load tokenizer
tok = AutoTokenizer.from_pretrained(base_path, local_files_only=True, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Load base model on CPU
base = AutoModelForCausalLM.from_pretrained(
    base_path,
    torch_dtype=torch.float32,
    local_files_only=True,
    trust_remote_code=True,
    device_map={"": "cpu"},
)

# Attach LoRA adapter
model = PeftModel.from_pretrained(base, str(adapter_dir), local_files_only=True)
model.eval()

def gen(prompt: str, max_new_tokens: int = 200, do_sample: bool = False):
    inputs = tok(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=0.0 if not do_sample else 0.7,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0], skip_special_tokens=True)

In [14]:
TEST_PROMPTS = [
    # Introduction & General Guidelines
    "Summarize the key billing policies from the Introduction and General Guidelines. End with [pp. 3–8, 11–16, 19–20A].",
    "List documentation requirements typically emphasized in the General Guidelines. End with [pp. 3–8].",

    # Evaluation & Management
    "Outline the E/M visit levels and high-level criteria relevant to the schedule. End with [pp. 27–28A].",
    "What are common pitfalls in E/M coding noted in the schedule? Provide brief bullets. End with [pp. 27–28A].",
]

for i, q in enumerate(TEST_PROMPTS, 1):
    print("\n" + "="*88)
    print(f"[{i}] Prompt:\n{q}\n")
    ans = gen(q, max_new_tokens=220, do_sample=False)
    print("Answer:\n" + ans)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



[1] Prompt:
Summarize the key billing policies from the Introduction and General Guidelines. End with [pp. 3–8, 11–16, 19–20A].



The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Answer:
Summarize the key billing policies from the Introduction and General Guidelines. End with [pp. 3–8, 11–16, 19–20A]. ### Key Billing Policies

{ "short": "The hospital has established various fee schedules for different services provided at its medical center or outpatient facilities. These include relative value unit (RVU) based rates, PC/TC split codes, and FUD codes., "medium": "The hospital has adopted fee schedules that provide for relative value unit (RVU) based rates, professional component/techinical component (PC/TC) split codes, and fair work unit (FUD) codes for various services rendered at its medical center or outpatient facilities., "long": [ { "bullet": "Services rendered at the hospital's medical center or outpatient facilities are subject to fee schedules approved by the chair of the Medical Services Committee (MSC). The fee schedules include relative value unit (RVU) based rates, professional component/techinical component (PC/TC) split codes, and fair work uni

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Answer:
List documentation requirements typically emphasized in the General Guidelines. End with [pp. 3–8]. ### Medical Treatment Guidelines

{ "short": "Treatment should be provided in accordance with applicable medical treatment guidelines adopted by the hospital or other appropriate authority. If no guidelines exist, treatment should be provided on a case-by-case basis after consultation with an appropriately qualified healthcare professional., "medium": "Treatment should be provided in accordance with applicable medical treatment guidelines adopted by the hospital or other appropriate authority. If no guidelines exist, treatment should be provided on a case-by-case basis after consultation with an appropriately qualified healthcare professional. The treatment plan must be reviewed and approved by a physician who has treated the patient previously., "long": [ { "heading": "Treatment of Injuries/Trauma, "bullet_list": [ - "For injuries/trauma that have undergone pre-authorization at 

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Answer:
Outline the E/M visit levels and high-level criteria relevant to the schedule. End with [pp. 27–28A]. - { "short": "The hospital outpatient and inpatient hospital use of services (HUISS) codes identify separately identifiable hospital services, procedures, or facilities provided at hospitals, nursing homes, or other institutional care providers., "medium": "Hospital outpatient and inpatient hospital use of services (HUISS) codes identify separately identifiable hospital services, procedures, or facilities provided at hospitals, nursing homes, or other institutional care providers. These codes apply whether or not hospital resources were used to provide the service(s). , "long": [ { "code": 100, "name": "No Services Rendered", "cyDescription": "Services rendered for which no charge has been or will be made. ", "thresholds": [] }, { "code": 101, "name": "Time Spent Without Services Rendered", "cyDescription": "Time spent without furnishing services that would have been charged if

### **11) Merge Adapter + Test Generation**

In [15]:
import subprocess
MERGED_DIR = ROOT / "outputs" / "lora_hf" / f"{DOC_ID}_merged"

args = [
    sys.executable, str(ROOT / "src/train/merge_lora.py"),
    "--base_model", MODEL_ID,
    "--lora_dir", str(OUT / "adapter"),
    "--out_dir", str(MERGED_DIR),
    "--local_files_only",
    "--trust_remote_code",
]
print("Merging:", " ".join(map(shlex.quote, args)))
print(subprocess.run(args, capture_output=True, text=True).stdout)

Merging: 'd:\Anaconda\envs\pdf-agent-2\python.exe' 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\src\train\merge_lora.py' --base_model 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\models\Qwen2.5-1.5B-Instruct' --lora_dir 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\outputs\lora_hf\New_York_State_Workers_Compensation_Medical_Fee\adapter' --out_dir 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\outputs\lora_hf\New_York_State_Workers_Compensation_Medical_Fee_merged' --local_files_only --trust_remote_code
[OK] merged model saved to: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\outputs\lora_hf\New_York_State_Workers_Compensation_Medical_Fee_merged



### **12) Test Prompt - Merged**

In [16]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(str(MERGED_DIR), local_files_only=True, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    str(MERGED_DIR),
    torch_dtype=torch.float32,
    local_files_only=True,
    trust_remote_code=True,
    device_map={"": "cpu"},
)
model.eval()

prompt = "What are common pitfalls in E/M coding noted in the schedule? Provide brief bullets. End with [pp. 27–28A]."
inputs = tok(prompt, return_tensors="pt")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=180, do_sample=False)
print(tok.decode(out[0], skip_special_tokens=True))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


What are common pitfalls in E/M coding noted in the schedule? Provide brief bullets. End with [pp. 27–28A]. { "category": "Coding, Medical", "subcategories": [ "Code Correctly", "Consider Exceptions" ] }.uela
- Misdiagnosis or misclassification can lead to inappropriate hospital charges.
- Failure to report services properly (e.g., PC/TC split incorrectly) can result in penalties or denied claims.
- Failing to identify and bill for previously scheduled procedures/services may cause errors.


### **6) Reranker Training**

In [4]:
OUT_RR = ROOT / "outputs" / "reranker" / DOC_ID
OUT_RR.mkdir(parents=True, exist_ok=True)

In [5]:
import json, itertools
fp = SFT_DIR / "train.jsonl"
print("showing 3 rows from", fp)
for line in itertools.islice(open(fp, "r", encoding="utf-8"), 3):
    obj = json.loads(line)
    print(list(obj.keys())[:10], "→ type:", type(obj))

showing 3 rows from D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sft\train.jsonl
['messages', 'response'] → type: <class 'dict'>
['messages', 'response'] → type: <class 'dict'>
['messages', 'response'] → type: <class 'dict'>


### **Build retrieval pairs**

In [9]:
import subprocess, sys, shlex
# IMPORTANT: pass the **directory** that holds *.jsonl (NOT the chunks.jsonl file)
CHUNKS_DIR = RUN_DIR / "chunks"          # already defined above
TRAIN_JSON = SFT_DIR / "train.jsonl"
DEV_JSON   = SFT_DIR / "dev.jsonl"
PAIRS_DIR  = RUN_DIR / "pairs"
PAIRS_DIR.mkdir(parents=True, exist_ok=True)

assert CHUNKS_DIR.is_dir(), f"{CHUNKS_DIR} is not a directory"
assert TRAIN_JSON.exists(), f"Missing {TRAIN_JSON}"

args = [
    sys.executable, str(ROOT / "src/train/retriever_pairs.py"),
    "--chunks_dir", str(CHUNKS_DIR),          # ← directory, not file
    "--train_jsonl", str(TRAIN_JSON),
    "--dev_jsonl",   str(DEV_JSON),
    "--out_dir",     str(PAIRS_DIR),
    "--topk", "30",
    "--neg_per_q", "4",
    "--min_query_len", "8",                   # relax for your short queries
]

print("Launching:", " ".join(map(shlex.quote, args)))
res = subprocess.run(args, capture_output=True, text=True)
print("STDOUT:\n", res.stdout)
print("STDERR:\n", res.stderr)
if res.returncode != 0:
    raise RuntimeError("retriever_pairs.py failed")

Launching: 'd:\Anaconda\envs\pdf-agent-2\python.exe' 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\src\train\retriever_pairs.py' --chunks_dir 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\chunks' --train_jsonl 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sft\train.jsonl' --dev_jsonl 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sft\dev.jsonl' --out_dir 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\pairs' --topk 30 --neg_per_q 4 --min_query_len 8
STDOUT:
 [info] loaded 1494 chunks
[stats] kept=15/15 | skipped_short=0 | page=0 | substr=0 | top1=15
[OK] wrote 15 train pairs → D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\pairs\train.pairs.jsonl
[stats] kept=1/1 | skipp

In [10]:
import json, itertools
train_pairs = PAIRS_DIR / "train.pairs.jsonl"
dev_pairs   = PAIRS_DIR / "dev.pairs.jsonl"

def peek_pairs(p: Path, n=3):
    if not p.exists():
        print("[MISSING]", p)
        return
    print(f"\n{p} (first {n})")
    for line in itertools.islice(open(p, "r", encoding="utf-8"), n):
        print(json.loads(line))

peek_pairs(train_pairs, 3)
peek_pairs(dev_pairs, 2)


D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\pairs\train.pairs.jsonl (first 3)
{'query': 'Summarize the section:\nHeading: Assembly Instructions > NEW CPT CODES\nSummarize as 12–15 detailed bullet points. End with [pp. 10–83].', 'positive': 'Assembly Instructions', 'negatives': ['Some services performed are not described by any CPT code. These services should be reported using an unlisted code and substantiating it by report as discussed in Surgery Ground Rule 10. The unlisted procedures and accompanying codes for surgery will be found at the end of the relevant section or subsection.', 'Some services performed are not described by any CPT code. These services should be reported using an unlisted code and substantiated by report as discussed in Surgery Ground Rule 10 above. The unlisted procedures and accompanying codes for surgery will be found at the end of the relevant section or subsection.', 'NEW CPT CODES', 'NEW CPT

In [11]:
import subprocess, sys, shlex

OUT_RR = ROOT / "outputs" / "reranker" / DOC_ID
OUT_RR.mkdir(parents=True, exist_ok=True)

# If you have a local cache of the CE model, it’ll stay fully offline:
LOCAL_CE = ROOT / "models" / "msmarco-minilm-l6-v2"
base_ce_id = str(LOCAL_CE if LOCAL_CE.exists() else "cross-encoder/ms-marco-MiniLM-L-6-v2")
print("[info] using base CE:", base_ce_id)

args = [
    sys.executable, str(ROOT / "src/train/train_reranker.py"),
    "--train_pairs", str(train_pairs),
    "--dev_pairs",   str(dev_pairs),
    "--base_ce",     "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "--local_base_dir", base_ce_id,
    "--out_dir",     str(OUT_RR),
    "--epochs",      "2",
    "--batch_size",  "8",
    "--lr",          "2e-5",
    "--max_len",     "384",
]
print("Launching:", " ".join(map(shlex.quote, args)))
proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
code = proc.wait()
print("\n[proc exit code]", code)
if code != 0:
    raise RuntimeError("Reranker training failed")

[info] using base CE: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\models\msmarco-minilm-l6-v2
Launching: 'd:\Anaconda\envs\pdf-agent-2\python.exe' 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\src\train\train_reranker.py' --train_pairs 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\pairs\train.pairs.jsonl' --dev_pairs 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\pairs\dev.pairs.jsonl' --base_ce cross-encoder/ms-marco-MiniLM-L-6-v2 --local_base_dir 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\models\msmarco-minilm-l6-v2' --out_dir 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\outputs\reranker\New_York_State_Workers_Compensation_Medical_Fee' --epochs 2 --batch_size 8 --lr 2e-5 --max_len 384
[info] base CE: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\models\msmarco-minilm-l6-v2
[info] train examples: 75
[info] total_steps=20 | war

### **7) Sanity-check retrieval (BM25 → CE rerank)**

In [12]:
import json, re
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

def _tok(s: str): return re.findall(r"[A-Za-z0-9_]+", (s or "").lower())

# Build BM25 over this run’s chunks
corpus = []
for fp in sorted(CHUNKS_DIR.glob("*.jsonl")):
    with open(fp, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            txt = obj.get("text") or obj.get("content") or ""
            if txt.strip(): corpus.append(txt)

print("Loaded chunks:", len(corpus))
bm25 = BM25Okapi([_tok(t) for t in corpus])

# Load trained reranker
reranker = CrossEncoder(str(OUT_RR), device="cpu")

def search(query: str, k_bm25=25, k_final=5):
    scores = bm25.get_scores(_tok(query))
    idxs = sorted(range(len(corpus)), key=lambda i: scores[i], reverse=True)[:k_bm25]
    cands = [corpus[i] for i in idxs]
    ce_scores = reranker.predict([[query, c] for c in cands])
    ranked = sorted(zip(cands, ce_scores), key=lambda x: x[1], reverse=True)
    return ranked[:k_final]

# Try domain-appropriate queries (you can replace these with your own)
probes = [
    "Summarize the key billing rules in the Introduction and General Guidelines.",
    "What documentation is required for Evaluation and Management visits?",
    "Give 4 bullets on Physical Medicine therapy coverage limits.",
]
for q in probes:
    print("\nQ:", q)
    for i, (txt, s) in enumerate(search(q), 1):
        print(f"\n#{i}  CE={float(s):.4f}\n{txt[:600]}…")

d:\Anaconda\envs\pdf-agent-2\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loaded chunks: 1494

Q: Summarize the key billing rules in the Introduction and General Guidelines.

#1  CE=3.6096
Note: Rules used by all provider in reporting their services are presented in the General Ground Rules in the Introduction and General Guidelines section.…

#2  CE=3.5665
Note: Rules used by all providers in reporting their services are presented in the General Ground Rules in the Introduction and General Guidelines section.…

#3  CE=2.3307
Introduction and General Guidelines…

#4  CE=2.3307
Introduction and General Guidelines…

#5  CE=2.3307
Introduction and General Guidelines…

Q: What documentation is required for Evaluation and Management visits?

#1  CE=3.6078
The necessity for such visits is infrequent in cases covered by the Workers' Compensation Law. When necessary, a statement setting forth the medical indications justifying such visits shall be submitted. Please refer to the Evaluation and Management section for coding of these services.…

#2  CE=3.6078
The neces

### **8) Build Hierarchy**

In [13]:
GRAPH_DIR = RUN_DIR / "graph"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT      :", ROOT)
print("CHUNKS_DIR:", CHUNKS_DIR)
print("GRAPH_DIR :", GRAPH_DIR)
print("Python    :", platform.python_version(), "| CPU threads:", os.cpu_count())

ROOT      : D:\IIT BBS\Job Resources\Business Optima\pdf-agent
CHUNKS_DIR: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\chunks
GRAPH_DIR : D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\graph
Python    : 3.11.13 | CPU threads: 8


In [14]:
import subprocess, shlex

args = [
    sys.executable, str(ROOT / "src/graph/build_hierarchical.py"),
    "--chunks_dir", str(CHUNKS_DIR),   # directory with *.jsonl
    "--out_dir",    str(GRAPH_DIR),
]
print("Launching:", " ".join(map(shlex.quote, args)))
proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
code = proc.wait()
print("\n[proc exit code]", code)
if code != 0:
    raise RuntimeError("Hierarchy build failed")

Launching: 'd:\Anaconda\envs\pdf-agent-2\python.exe' 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\src\graph\build_hierarchical.py' --chunks_dir 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\chunks' --out_dir 'D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\graph'
[OK] built hierarchy: 2 nodes from 1363 chunks
[OK] wrote: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\graph\graph\hierarchy.json
[OK] wrote: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\graph\graph\node_texts.jsonl

[proc exit code] 0


In [15]:
import json, itertools

HIER_JSON   = GRAPH_DIR / "graph" / "hierarchy.json"
NODE_TEXTS  = GRAPH_DIR / "graph" / "node_texts.jsonl"

hier = json.loads(HIER_JSON.read_text(encoding="utf-8"))
print("#nodes:", hier.get("n_nodes"), "| #chunks:", hier.get("n_chunks"))
print("Sample node record keys:", list(hier["nodes"][0].keys()))

print("\nFirst 3 node_texts.jsonl rows:")
for i, line in enumerate(open(NODE_TEXTS, "r", encoding="utf-8")):
    print(json.loads(line))
    if i == 2: break

#nodes: 2 | #chunks: 1363
Sample node record keys: ['id', 'name', 'level', 'parent', 'children', 'chunk_ids']

First 3 node_texts.jsonl rows:
{'node_id': 'SEC-00001', 'name': 'MISC (no § heading detected)', 'level': 1, 'text': "January 1, 2020\n\nDear Customer:\n\nThe State of New York has revised the Official New York State Workers' Compensation Medical Fee Schedule effective January 1, 2020. The enclosed pages will update the 2018 edition (effective April 1, 2019) with the updates effective January 1, 2020.\n\nThe new Official New York State Workers' Compensation Acupuncture and Physical &amp; Occupational Therapy Fee Schedules booklet is not included in this update. This booklet is available separately by calling Optum360 at 1.800.464.3649, option 1.\n\nSincerely,\n\nOptum360\n\n<!-- image -->\n\nLWCNY18R\n\nAssembly Instructions\n\nOfficial New York State Workers' Compensation Medical Fee Schedule\n\nPlease follow these instructions to assemble your book. Remove pages in the OLD Pa

### **9) Node search: BM25 over nodes + trained reranker (fallback-safe)**

In [16]:
import json, re
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

NODE_TEXTS  = GRAPH_DIR / "graph" / "node_texts.jsonl"
node_records = [json.loads(line) for line in open(NODE_TEXTS, "r", encoding="utf-8")]
node_texts  = [r["text"] for r in node_records]
node_names  = [r["name"] for r in node_records]
node_ids    = [r["node_id"] for r in node_records]

def _tok(s: str):
    return re.findall(r"[A-Za-z0-9_]+", (s or "").lower())

bm25_nodes = BM25Okapi([_tok(t) for t in node_texts])

# Try to load THIS RUN’s reranker; fall back to BM25-only if unavailable
from pathlib import Path
OUT_RR = ROOT / "outputs" / "reranker" / DOC_ID
def _load_reranker_or_none(path: Path):
    try:
        if path.exists():
            return CrossEncoder(str(path), device="cpu")
    except Exception as e:
        print("[warn] reranker load failed:", e)
    return None

reranker = _load_reranker_or_none(OUT_RR)
print("Reranker:", str(OUT_RR) if reranker else "(none: BM25-only)")

def hier_search(query: str, k_nodes=20, k_final=5):
    # 1) BM25 candidates
    scores = bm25_nodes.get_scores(_tok(query))
    idxs = sorted(range(len(node_texts)), key=lambda i: scores[i], reverse=True)[:k_nodes]

    # 2) Optional CE rerank
    if reranker:
        ce = reranker.predict([[query, node_texts[i]] for i in idxs])
        ranked = sorted(zip(idxs, ce), key=lambda x: x[1], reverse=True)[:k_final]
    else:
        ranked = [(i, scores[i]) for i in idxs[:k_final]]

    # 3) Return compact results
    out = []
    for i, sc in ranked:
        snippet = (node_texts[i][:900] + "…") if len(node_texts[i]) > 900 else node_texts[i]
        out.append({
            "node_id": node_ids[i],
            "name":    node_names[i],
            "score":   float(sc),
            "snippet": snippet
        })
    return out

Reranker: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\outputs\reranker\New_York_State_Workers_Compensation_Medical_Fee


In [17]:
probes = [
    "Summarize the key billing rules in the Introduction and General Guidelines.",
    "What documentation is required for Evaluation and Management visits?",
    "Give 4 bullets on Physical Medicine therapy coverage limits.",
    "Briefly summarize core rules in the Surgery section.",
    "What counts as reimbursable Radiology documentation?",
    "List key items in Pathology and Laboratory billing requirements.",
]

for q in probes:
    print("\nQ:", q)
    for i, r in enumerate(hier_search(q), 1):
        print(f"\n[{i}] node={r['node_id']}  score={r['score']:.3f}\n{r['name']}\n{r['snippet']}")


Q: Summarize the key billing rules in the Introduction and General Guidelines.

[1] node=SEC-00001  score=-10.739
MISC (no § heading detected)
January 1, 2020

Dear Customer:

The State of New York has revised the Official New York State Workers' Compensation Medical Fee Schedule effective January 1, 2020. The enclosed pages will update the 2018 edition (effective April 1, 2019) with the updates effective January 1, 2020.

The new Official New York State Workers' Compensation Acupuncture and Physical &amp; Occupational Therapy Fee Schedules booklet is not included in this update. This booklet is available separately by calling Optum360 at 1.800.464.3649, option 1.

Sincerely,

Optum360

<!-- image -->

LWCNY18R

Assembly Instructions

Official New York State Workers' Compensation Medical Fee Schedule

Please follow these instructions to assemble your book. Remove pages in the OLD Pages column and replace with pages in the NEW Pages column.

<!-- image -->

Effective 4/1/2019 Revisions

In [18]:
from pathlib import Path
import json, collections, itertools, re

NODE_TEXTS = RUN_DIR / "graph" / "graph" / "node_texts.jsonl"
HIER_JSON  = RUN_DIR / "graph" / "graph" / "hierarchy.json"

def peek_nodes(node_texts_path: Path, n=5):
    if not node_texts_path.exists():
        print("[MISSING]", node_texts_path); return
    print("[peek]", node_texts_path)
    for i, ln in enumerate(open(node_texts_path, "r", encoding="utf-8")):
        print(json.loads(ln))
        if i+1 >= n: break

if HIER_JSON.exists():
    hier = json.loads(HIER_JSON.read_text(encoding="utf-8"))
    print("#nodes:", hier.get("n_nodes"), "| #chunks:", hier.get("n_chunks"))
    print("node[0] keys:", list(hier["nodes"][0].keys()))
else:
    print("[MISSING]", HIER_JSON)

peek_nodes(NODE_TEXTS, 3)

#nodes: 2 | #chunks: 1363
node[0] keys: ['id', 'name', 'level', 'parent', 'children', 'chunk_ids']
[peek] D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\graph\graph\node_texts.jsonl
{'node_id': 'SEC-00001', 'name': 'MISC (no § heading detected)', 'level': 1, 'text': "January 1, 2020\n\nDear Customer:\n\nThe State of New York has revised the Official New York State Workers' Compensation Medical Fee Schedule effective January 1, 2020. The enclosed pages will update the 2018 edition (effective April 1, 2019) with the updates effective January 1, 2020.\n\nThe new Official New York State Workers' Compensation Acupuncture and Physical &amp; Occupational Therapy Fee Schedules booklet is not included in this update. This booklet is available separately by calling Optum360 at 1.800.464.3649, option 1.\n\nSincerely,\n\nOptum360\n\n<!-- image -->\n\nLWCNY18R\n\nAssembly Instructions\n\nOfficial New York State Workers' Compensation Medi

### **Book/section-aware fallback hierarchy**

In [20]:
# --- Build a hierarchy tailored to the fee-schedule books/sections ---
from pathlib import Path
import json, re
from collections import defaultdict

# Reuse run-scoped vars from earlier cells:
# ROOT, RUN_DIR, DOC_ID, CHUNKS_DIR already exist
CHUNKS_JSONL = RUN_DIR / "chunks" / "chunks.jsonl"
GRAPH_ROOT   = RUN_DIR / "graph" / "graph"
GRAPH_ROOT.mkdir(parents=True, exist_ok=True)

BOOKS = [
    "Medical Fee Schedule",
    "Behavioral Health Fee Schedule",
    "Chiropractic Fee Schedule",
    "Podiatry Fee Schedule",
]

SUBSECTIONS = [
    "Introduction and General Guidelines",
    "Evaluation and Management",
    "Surgery",
    "Radiology",
    "Pathology and Laboratory",
    "Medicine",
    "Physical Medicine",
    "Contents",
    "Title/Disclaimer pages",
]

ALIASES = {
    "Evaluation and Management": [r"\bE\s*/\s*M\b", r"Evaluation\s*&\s*Management", r"Evaluation\s+and\s+Management"],
    "Pathology and Laboratory":  [r"Pathology\s*&\s*Laboratory", r"Pathology\s+and\s+Laboratory"],
}

FRONT_MATTER_PAT = re.compile(
    r"(assembly instructions|title/disclaimer pages|contents\b|^dear customer|"
    r"effective\s+[0-9/]+|official .* workers[’'] compensation .* fee schedule)",
    re.I
)

def _extract_heading_parts(meta: dict) -> list[str]:
    """Return a normalized list of heading parts from metadata."""
    hp = (meta or {}).get("heading_path")
    out = []
    if isinstance(hp, str):
        out = [p.strip() for p in hp.split(">") if p and p.strip()]
    elif isinstance(hp, list):
        for el in hp:
            if isinstance(el, str):
                t = el.strip()
                if t: out.append(t)
            elif isinstance(el, dict):
                t = (el.get("title") or el.get("name") or el.get("heading") or "").strip()
                if t: out.append(t)
    # fallbacks
    for k in ("heading", "section", "chapter"):
        v = (meta or {}).get(k)
        if isinstance(v, str) and v.strip():
            out.append(v.strip())
    return out

def top_from_heading_path(meta: dict) -> tuple[str|None, str|None]:
    parts = _extract_heading_parts(meta)
    book = None
    section = None
    # book via canonical match
    for p in parts:
        for b in BOOKS:
            if b.lower() in p.lower():
                book = b
                break
        if book: break
    # section via canonical + alias patterns
    for p in parts:
        for s in SUBSECTIONS:
            if s.lower() in p.lower():
                section = s
                break
        if section: break
    if not section:
        for p in parts:
            for sec, pats in ALIASES.items():
                if any(re.search(rx, p, re.I) for rx in pats):
                    section = sec
                    break
            if section: break
    return book, section

def find_book_in_text(text: str) -> str|None:
    low = (text or "").lower()
    for b in BOOKS:
        if b.lower() in low:
            return b
    return None

def find_section_in_text(text: str) -> str|None:
    low = (text or "").lower()
    for s in SUBSECTIONS:
        if s.lower() in low:
            return s
    for sec, pats in ALIASES.items():
        if any(re.search(rx, low, re.I) for rx in pats):
            return sec
    return None

def classify_chunk(meta: dict, text: str) -> tuple[str, str]:
    if FRONT_MATTER_PAT.search(text or ""):
        return ("Front Matter", "front_matter")
    b_hp, s_hp = top_from_heading_path(meta)
    b_tx = find_book_in_text(text or "")
    s_tx = find_section_in_text(text or "")
    book = b_hp or b_tx
    section = s_hp or s_tx
    if book and section:
        return (f"{book} :: {section}", "hp/text both")
    if section:
        return (section, "section only")
    if book:
        return (f"{book} :: (Unlabeled Section)", "book only")
    return ("Other", "fallback")

groups: dict[str, list[str]] = defaultdict(list)
counts = defaultdict(int)

with open(CHUNKS_JSONL, "r", encoding="utf-8") as f:
    for ln in f:
        obj = json.loads(ln)
        txt = obj.get("text") or obj.get("content") or ""
        if not (txt and txt.strip()):
            continue
        group, why = classify_chunk(obj.get("metadata") or {}, txt)
        if group == "Front Matter":
            continue
        groups[group].append(txt)
        counts[group] += 1

nodes = []
node_texts = []
node_id_seq = 1
n_chunks_total = sum(counts.values())

for name, texts in sorted(groups.items(), key=lambda kv: (-len(kv[1]), kv[0].lower())):
    nid = f"SEC-{node_id_seq:05d}"
    node_id_seq += 1
    joined = "\n\n".join(texts)
    nodes.append({"node_id": nid, "name": name, "level": 0, "n_chunks": len(texts)})
    node_texts.append({"node_id": nid, "name": name, "text": joined})

HIER_JSON = GRAPH_ROOT / "hierarchy.json"
NODE_TEXTS = GRAPH_ROOT / "node_texts.jsonl"

with open(HIER_JSON, "w", encoding="utf-8") as f:
    json.dump({"n_nodes": len(nodes), "n_chunks": n_chunks_total, "nodes": nodes},
              f, ensure_ascii=False, indent=2)
with open(NODE_TEXTS, "w", encoding="utf-8") as f:
    for r in node_texts:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"[OK] built hierarchy → {HIER_JSON}")
print(f"[OK] node texts      → {NODE_TEXTS}")
print("Top groups:")
for n in nodes[:12]:
    print(f"  - {n['name']}: {n['n_chunks']} chunks")


[OK] built hierarchy → D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\graph\graph\hierarchy.json
[OK] node texts      → D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\graph\graph\node_texts.jsonl
Top groups:
  - Other: 1195 chunks
  - Evaluation and Management: 70 chunks
  - Surgery: 64 chunks
  - Medicine: 63 chunks
  - Pathology and Laboratory: 17 chunks
  - Medical Fee Schedule :: (Unlabeled Section): 14 chunks
  - Behavioral Health Fee Schedule :: (Unlabeled Section): 12 chunks
  - Podiatry Fee Schedule :: (Unlabeled Section): 12 chunks
  - Introduction and General Guidelines: 9 chunks
  - Chiropractic Fee Schedule :: (Unlabeled Section): 8 chunks
  - Contents: 4 chunks
  - Radiology: 3 chunks


### **Querying the hierarchy nodes (BM25 → CE reranker)**

In [21]:
# --- Search hierarchy nodes with trained reranker for this DOC_ID ---
import json, re
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

NODE_TEXTS = RUN_DIR / "graph" / "graph" / "node_texts.jsonl"
recs = [json.loads(l) for l in open(NODE_TEXTS, "r", encoding="utf-8")]
node_texts = [r["text"] for r in recs]
node_names = [r["name"] for r in recs]
node_ids   = [r["node_id"] for r in recs]

def tok(s): return re.findall(r"[A-Za-z0-9_]+", (s or "").lower())
bm25_nodes = BM25Okapi([tok(t) for t in node_texts])

OUT_RR = ROOT / "outputs" / "reranker" / DOC_ID
reranker = CrossEncoder(str(OUT_RR), device="cpu") if OUT_RR.exists() else None
if reranker is None:
    print("[warn] trained reranker not found; using BM25 only.")

def hier_search(query: str, k_nodes=20, k_final=5):
    scores = bm25_nodes.get_scores(tok(query))
    idxs = sorted(range(len(node_texts)), key=lambda i: scores[i], reverse=True)[:k_nodes]
    if reranker:
        ce = reranker.predict([[query, node_texts[i]] for i in idxs])
        ranked = sorted(zip(idxs, ce), key=lambda x: x[1], reverse=True)[:k_final]
    else:
        ranked = [(i, scores[i]) for i in idxs[:k_final]]
    out = []
    for i, sc in ranked:
        snip = (node_texts[i][:900] + "…") if len(node_texts[i]) > 900 else node_texts[i]
        out.append((node_ids[i], node_names[i], float(sc), snip))
    return out

probes = [
    "Summarize key billing rules in the Medical Fee Schedule – Introduction and General Guidelines.",
    "What documentation is required for Evaluation and Management (E/M) visits in the Medical Fee Schedule?",
    "Give 4 bullets on Physical Medicine therapy coverage limits in the Medical Fee Schedule.",
    "Briefly summarize core Surgery section rules in the Medical Fee Schedule.",
    "What counts as reimbursable Radiology documentation in the Podiatry Fee Schedule?",
    "List key items in Pathology and Laboratory billing requirements in the Podiatry Fee Schedule.",
    "Summarize the Behavioral Health Fee Schedule – Introduction and General Guidelines.",
    "Chiropractic Fee Schedule: list Physical Medicine coverage notes.",
]

for q in probes:
    print("\nQ:", q)
    for i, (nid, name, sc, snip) in enumerate(hier_search(q), 1):
        print(f"\n[{i}] node={nid}  score={sc:.3f}\n{name}\n{snip[:650]}…")



Q: Summarize key billing rules in the Medical Fee Schedule – Introduction and General Guidelines.

[1] node=SEC-00009  score=3.819
Introduction and General Guidelines
Introduction and General Guidelines

'BR' in the Relative Value column indicates that the relative value unit of this service is to be determined 'by report.' Pertinent information concerning the nature, extent, and need for the procedure or service, the time, the skill, and equipment necessary , etc., is to be furnished. A detailed clinical record is not necessary. See the Ground Rules in the Introduction and General Guidelines section for a complete explanation of 'by report' procedures.

Introduction and General Guidelines

The accompanying instructions and ground rules explain the application of these procedure descriptors and relative va…

[2] node=SEC-00006  score=1.049
Medical Fee Schedule :: (Unlabeled Section)
Please follow these instructions to assemble your book. Remove pages in the OLD Pages column and replac

In [31]:
# ========= RAG EVAL — paths & dev set =========
from pathlib import Path
import json, re, statistics
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# Reuse earlier vars:
# ROOT, RUN_DIR, DOC_ID, CHUNKS_DIR, GRAPH_DIR (RUN_DIR / "graph" / "graph")
PAIRS_DIR = RUN_DIR / "pairs"
DEV = PAIRS_DIR / "dev.pairs.jsonl"
assert DEV.exists(), f"Missing {DEV}"

dev_pairs = [json.loads(x) for x in open(DEV, "r", encoding="utf-8")]

# helpful tokenizer
def _tok(s: str): return re.findall(r"[A-Za-z0-9_]+", (s or "").lower())

# trained reranker (already trained above)
OUT_RR = ROOT / "outputs" / "reranker" / DOC_ID
ce = CrossEncoder(str(OUT_RR), device="cpu")


[OK] wrote 14 node membership rows → D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\graph\graph\node_members.jsonl
[info] hierarchy.json n_chunks already aligned


In [60]:
# --- PATCH: hierarchical retriever that uses node_members.jsonl ---

from __future__ import annotations
import json, re
from pathlib import Path
from typing import List, Dict, Any, Tuple

from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

def _tok(s: str) -> list[str]:
    return re.findall(r"[A-Za-z0-9_]+", (s or "").lower())

def _resolve_graph_dir(gdir: Path) -> Path:
    """Accept either .../graph or .../graph/graph and return the folder that contains the JSONLs."""
    gdir = Path(gdir)
    if (gdir / "node_texts.jsonl").exists() and (gdir / "hierarchy.json").exists():
        return gdir
    if (gdir / "graph" / "node_texts.jsonl").exists():
        return gdir / "graph"
    raise FileNotFoundError(f"node_texts.jsonl not found under {gdir}")

def load_graph(graph_dir: Path) -> tuple[list[dict], dict, dict]:
    """
    Returns:
      node_records: list of {node_id, name, text}
      hier_by_id  : {node_id -> node_dict}
      node2chunks : {node_id -> [chunk_id, ...]} from node_members.jsonl
    """
    graph_dir = _resolve_graph_dir(graph_dir)

    node_records = [json.loads(x) for x in open(graph_dir / "node_texts.jsonl", "r", encoding="utf-8")]
    hier = json.loads((graph_dir / "hierarchy.json").read_text(encoding="utf-8"))
    # Some builds keep "id" == "node_id" — normalize:
    for n in hier.get("nodes", []):
        if "id" not in n and "node_id" in n:
            n["id"] = n["node_id"]
    hier_by_id = {n["id"]: n for n in hier["nodes"]}

    # Memberships live here:
    node2chunks: dict[str, list[str]] = {}
    nm_path = graph_dir / "node_members.jsonl"
    if nm_path.exists():
        for line in open(nm_path, "r", encoding="utf-8"):
            row = json.loads(line)
            nid = row.get("node_id")
            cids = row.get("chunk_ids") or []
            if nid:
                node2chunks[nid] = cids
    else:
        # Fallback (shouldn't happen for your run)
        for n in hier["nodes"]:
            node2chunks[n["id"]] = n.get("chunk_ids") or []

    return node_records, hier_by_id, node2chunks

def _extract_pages_from_meta(meta: dict) -> list[int]:
    # explicit
    p = meta.get("pages") or meta.get("page")
    if isinstance(p, list):
        out = []
        for v in p:
            try: out.append(int(v))
            except Exception: pass
        return out
    if isinstance(p, (int, float)):
        v = int(p)
        return [v, v]

    # start/end
    for k in ("page_start", "start_page", "pageStart"):
        if k in meta:
            try:
                p1 = int(meta.get(k))
                p2 = int(meta.get("page_end", meta.get("end_page", meta.get("pageEnd", p1))))
                return [p1, p2]
            except Exception:
                pass

    # other common singletons
    for k in ("page_idx", "page_no", "pageNumber"):
        if k in meta:
            try:
                v = int(meta[k])
                return [v, v]
            except Exception:
                pass

    return []

def _normalize_section(obj: dict) -> str:
    meta = obj.get("metadata") or {}
    hp = meta.get("heading_path")
    if isinstance(hp, list):
        hp = " > ".join([str(x).strip() for x in hp if str(x).strip()])
    elif isinstance(hp, str):
        hp = hp.strip()
    else:
        hp = ""
    return hp or (obj.get("section") or "")

def load_chunks(chunks_dir: Path) -> dict[str, dict]:
    """
    Keyed by chunk id. Extracts text, section (from heading_path), and pages (robust).
    """
    id2chunk: dict[str, dict] = {}
    for fp in sorted(Path(chunks_dir).glob("*.jsonl")):
        with open(fp, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                except Exception:
                    continue
                cid = obj.get("id") or obj.get("chunk_id")
                txt = obj.get("text") or obj.get("content") or ""
                if not cid or not isinstance(txt, str) or not txt.strip():
                    continue
                meta = obj.get("metadata") or {}
                section = _normalize_section(obj)
                pages = _extract_pages_from_meta(meta)
                id2chunk[cid] = {
                    "id": cid,
                    "text": txt,
                    "section": section,
                    "pages": pages,
                }
    return id2chunk

class GraphRetriever:
    """
    BM25 over node texts -> CE rerank nodes -> BM25 within each node's member chunks (with slight heading boost) -> CE final rerank.
    """
    def __init__(self, chunks_dir: str | Path, graph_dir: str | Path, reranker_dir: str | Path):
        self.chunks_dir = Path(chunks_dir)
        self.graph_dir  = Path(graph_dir)
        self.reranker   = CrossEncoder(str(reranker_dir), device="cpu")

        self.node_records, self.hier_by_id, self.node2chunks = load_graph(self.graph_dir)
        self.node_texts  = [r["text"] for r in self.node_records]
        self.node_ids    = [r["node_id"] for r in self.node_records]
        self.node_names  = [r["name"]    for r in self.node_records]

        self.bm25_nodes  = BM25Okapi([_tok(t) for t in self.node_texts])
        self.id2chunk    = load_chunks(self.chunks_dir)

    def _best_nodes(self, query: str, k_nodes: int, k_final_nodes: int) -> list[int]:
        scores = self.bm25_nodes.get_scores(_tok(query))
        idxs   = sorted(range(len(self.node_texts)), key=lambda i: scores[i], reverse=True)[:k_nodes]
        if not idxs:
            return []
        ce = self.reranker.predict([[query, self.node_texts[i]] for i in idxs])
        reranked = sorted(zip(idxs, ce), key=lambda x: x[1], reverse=True)[:k_final_nodes]
        return [i for i, _ in reranked]

    def _best_chunks_from_nodes(self, query: str, node_idxs: list[int],
                                k_each_node: int, k_final_chunks: int) -> list[dict]:
        cands: list[dict] = []
        for ni in node_idxs:
            node_id = self.node_ids[ni]
            node_name = self.node_names[ni] if ni < len(self.node_names) else node_id
            member_ids = [cid for cid in self.node2chunks.get(node_id, []) if cid in self.id2chunk]
            if not member_ids:
                continue

            # lightweight local BM25 with heading boost
            sub_texts = [f"{node_name}\n{self.id2chunk[cid]['text']}" for cid in member_ids]
            bm25_local = BM25Okapi([_tok(t) for t in sub_texts])
            order = sorted(range(len(member_ids)),
                           key=lambda i: bm25_local.get_scores(_tok(query))[i],
                           reverse=True)[:k_each_node]
            for j in order:
                cid = member_ids[j]
                ch  = self.id2chunk[cid]
                cands.append({
                    "chunk_id": cid,
                    "node_id": node_id,
                    "node_name": node_name,
                    "text": ch["text"],
                    "pages": ch["pages"],
                    "section": ch["section"],
                })

        if not cands:
            return []
        ce = self.reranker.predict([[query, c["text"]] for c in cands])
        for c, s in zip(cands, ce):
            c["score"] = float(s)
        cands.sort(key=lambda x: x["score"], reverse=True)
        return cands[:k_final_chunks]

    def search(self, query: str,
               k_nodes: int = 60, k_final_nodes: int = 10,
               k_each_node: int = 16, k_final_chunks: int = 10) -> list[dict]:
        node_idxs = self._best_nodes(query, k_nodes=k_nodes, k_final_nodes=k_final_nodes)
        if not node_idxs:
            return []
        return self._best_chunks_from_nodes(query, node_idxs, k_each_node, k_final_chunks)


In [61]:
# Paths you already had
DOC_ID      = "New_York_State_Workers_Compensation_Medical_Fee"
CHUNKS_DIR  = RUN_DIR / "chunks"
GRAPH_DIR   = RUN_DIR / "graph" / "graph"      # ok; resolver also handles parent
RERANKER_DIR= ROOT / "outputs" / "reranker" / DOC_ID

hier_retr = GraphRetriever(CHUNKS_DIR, GRAPH_DIR, RERANKER_DIR)
print("Patched hierarchical retriever ready.\n")

probes = [
    "Summarize the key billing rules in the Introduction and General Guidelines.",
    "What documentation is required for Evaluation and Management visits?",
    "Give 4 bullets on Physical Medicine therapy coverage limits.",
]
for q in probes:
    hits = hier_retr.search(q, k_nodes=50, k_final_nodes=8, k_each_node=14, k_final_chunks=5)
    print(f"Q: {q}\n#hits: {len(hits)}")
    for i, h in enumerate(hits, 1):
        print(f"  [{i}] {h.get('section') or '(untitled)'} {h.get('pages')}")
    print()

Patched hierarchical retriever ready.

Q: Summarize the key billing rules in the Introduction and General Guidelines.
#hits: 5
  [1] Assembly Instructions > PHYSICAL MEDICINE GROUND RULES [45, 45]
  [2] Assembly Instructions > PATHOLOGY AND LABORATORY GROUND RULES [37, 37]
  [3] Assembly Instructions > SPECIALTY CLASSIFICATIONS [2, 2]
  [4] Assembly Instructions > Introduction and General Guidelines [2, 2]
  [5] Assembly Instructions > Introduction and General Guidelines [2, 2]

Q: What documentation is required for Evaluation and Management visits?
#hits: 5
  [1] Assembly Instructions > 9. Home Visits [86, 86]
  [2] Assembly Instructions > 9. Home Visits [86, 86]
  [3] Assembly Instructions > 27 Multiple Outpatient Hospital E/M Encounters on the Same Date (This CPT modifier is for use by Ambulatory Surgery Center (ASC) and Hospital [18, 18]
  [4] Assembly Instructions > Multiple Outpatient Hospital E/M Encounters on the Same Date (This CPT modifier is for use by Ambulatory Surgery Cen

In [62]:
import json, statistics

dev_path = RUN_DIR / "pairs" / "dev.pairs.jsonl"
if dev_path.exists():
    dev_pairs = [json.loads(x) for x in open(dev_path, "r", encoding="utf-8")]
else:
    dev_pairs = []

def eval_retrieval_hier(pairs, retr: GraphRetriever,
                        k_nodes=50, k_final_nodes=8, k_each_node=14, k_final_chunks=10):
    mrr10 = top1 = recall10 = 0.0
    ranks = []
    for row in pairs:
        q = row["query"]
        pos_chunk_id = (row.get("meta") or {}).get("pos_chunk_id")
        pos_hint = (row.get("positive") or "").strip()

        hits = retr.search(q, k_nodes, k_final_nodes, k_each_node, k_final_chunks)

        rank = None
        for i, h in enumerate(hits, 1):
            if pos_chunk_id and h["chunk_id"] == pos_chunk_id:
                rank = i; break
            if not pos_chunk_id and pos_hint and pos_hint[:40] in h["text"]:
                rank = i; break

        if rank is not None:
            ranks.append(rank)
            if rank == 1: top1 += 1
            if rank <= 10: recall10 += 1
            mrr10 += 1.0 / rank

    n = len(pairs) or 1
    return {
        "n": len(pairs),
        "top1": top1 / n,
        "recall@10": recall10 / n,
        "mrr@10": mrr10 / n,
        "median_rank": (statistics.median(ranks) if ranks else None),
    }

print("[Hierarchical] dev metrics:", eval_retrieval_hier(dev_pairs, hier_retr))

[Hierarchical] dev metrics: {'n': 1, 'top1': 1.0, 'recall@10': 1.0, 'mrr@10': 1.0, 'median_rank': 1}


In [63]:
# --- Core prompt builder + model loader ---

from pathlib import Path
import sys, os, json, time, platform
from datetime import datetime

CWD  = Path.cwd().resolve()
ROOT = CWD if (CWD / "src").exists() else CWD.parent
if str(ROOT) not in sys.path: sys.path.append(str(ROOT))

DOC_ID      = "New_York_State_Workers_Compensation_Medical_Fee"
RUN_DIR     = ROOT / "data" / "runs" / DOC_ID
CHUNKS_DIR  = RUN_DIR / "chunks"
GRAPH_DIR   = RUN_DIR / "graph" / "graph"
RERANKER_DIR= ROOT / "outputs" / "reranker" / DOC_ID
LORA_OUT    = ROOT / "outputs" / "lora_hf" / DOC_ID
MERGED_DIR  = ROOT / "outputs" / "lora_hf" / f"{DOC_ID}_merged"
BASE_QWEN   = ROOT / "models" / "Qwen2.5-1.5B-Instruct"

SESS_DIR    = RUN_DIR / "sessions"
SESS_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT      :", ROOT)
print("DOC_ID    :", DOC_ID)
print("GRAPH_DIR :", GRAPH_DIR)
print("RERANKER  :", RERANKER_DIR)
print("SESS_DIR  :", SESS_DIR)

# Use hierarchical retriever you just patched/instantiated earlier:
# hier_retr = GraphRetriever(CHUNKS_DIR, GRAPH_DIR, RERANKER_DIR)

def build_system_prompt(doc_id: str) -> str:
    return (
        "You are a focused assistant for a single PDF (New York State Workers’ Compensation Medical Fee Schedules). "
        "Answer STRICTLY from the provided contexts. If the contexts are insufficient, say so. "
        "Prefer concise, practical wording. Include bracketed citations from the context tail exactly as given "
        "(e.g., [pp. 21–22] or [chunk <id>]). Do not invent page numbers.\n"
        f"Document ID: {doc_id}\n"
        "When summarizing rules, prefer short bullets; when answering factual questions, give a direct answer first."
    )

def build_user_prompt(question: str, contexts: list[str]) -> str:
    ctx = "\n\n".join(f"--- Context {i+1} ---\n{c}" for i, c in enumerate(contexts))
    instructions = (
        "Instructions:\n"
        "1) Use only the facts in the contexts.\n"
        "2) Keep citations in-square-brackets as they appear at the end of the relevant sentence(s).\n"
        "3) If unsure, say you don’t have enough info.\n"
    )
    return f"{instructions}\nQuestion: {question}\n\n{ctx}\n\nAnswer:"

# --- Load the core model (prefer MERGED; otherwise base+adapter) ---
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

os.environ["TRANSFORMERS_OFFLINE"] = "1"   # for your offline setup
LOCAL_ONLY = True

USE_MERGED = MERGED_DIR.exists()
print("Using merged model?" , USE_MERGED)

if USE_MERGED:
    tok = AutoTokenizer.from_pretrained(str(MERGED_DIR), local_files_only=LOCAL_ONLY, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    core_model = AutoModelForCausalLM.from_pretrained(
        str(MERGED_DIR), torch_dtype=torch.float32, local_files_only=LOCAL_ONLY,
        trust_remote_code=True, device_map={"": "cpu"},
    ).eval()
else:
    tok = AutoTokenizer.from_pretrained(str(BASE_QWEN), local_files_only=LOCAL_ONLY, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(
        str(BASE_QWEN), torch_dtype=torch.float32, local_files_only=LOCAL_ONLY,
        trust_remote_code=True, device_map={"": "cpu"},
    )
    adapter_dir = LORA_OUT / "adapter"
    core_model = PeftModel.from_pretrained(base, str(adapter_dir), local_files_only=True).eval()

def generate_core(prompt: str, max_new_tokens=320, temperature=0.0) -> str:
    inputs = tok(prompt, return_tensors="pt")
    with torch.no_grad():
        out = core_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0.0),
            temperature=temperature if temperature > 0 else None,
            top_p=0.9 if temperature > 0 else None,
            eos_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0], skip_special_tokens=True)

ROOT      : D:\IIT BBS\Job Resources\Business Optima\pdf-agent
DOC_ID    : New_York_State_Workers_Compensation_Medical_Fee
GRAPH_DIR : D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\graph\graph
RERANKER  : D:\IIT BBS\Job Resources\Business Optima\pdf-agent\outputs\reranker\New_York_State_Workers_Compensation_Medical_Fee
SESS_DIR  : D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sessions
Using merged model? True


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [64]:
# --- Optional polisher via Ollama (keeps citations) ---
import os, requests

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "llama3:instruct")  # or any local fast model

def polish_answer(raw: str) -> str:
    """
    Light rewrite for clarity; MUST preserve bracketed citations exactly.
    If Ollama is not running, just return raw.
    """
    try:
        prompt = (
            "Rewrite the answer to be clear and compact without changing facts. "
            "Do NOT remove or alter any bracketed citations like [pp. 21–22] or [chunk ...].\n\n"
            f"Answer:\n{raw}\n\nPolished:"
        )
        r = requests.post(
            f"{OLLAMA_HOST.rstrip('/')}/api/generate",
            json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
            timeout=30,
        )
        r.raise_for_status()
        txt = r.json().get("response") or ""
        return txt.strip() or raw
    except Exception:
        return raw

In [65]:
# --- RAG answerer using hierarchical retriever ---

from time import time as _time

def answer_with_rag(question: str,
                    k_nodes=50, k_final_nodes=8, k_each_node=14, k_final_chunks=6,
                    polish=True) -> dict:
    # Retrieve
    hits = hier_retr.search(question, k_nodes, k_final_nodes, k_each_node, k_final_chunks)

    # Build contexts for the prompt (uses your earlier prepare_contexts)
    ctx_texts = prepare_contexts(hits, max_chars=1200)

    sys_msg = build_system_prompt(DOC_ID)
    user_msg = build_user_prompt(question, ctx_texts)
    full_prompt = f"<|system|>\n{sys_msg}\n</|system|>\n<|user|>\n{user_msg}\n</|user|>\n<|assistant|>\n"

    t0 = _time()
    raw = generate_core(full_prompt, max_new_tokens=320, temperature=0.0).strip()
    gen_ms = int((_time() - t0) * 1000)

    # Some Qwen chat templates will echo the prompt—strip if needed:
    if raw.startswith(user_msg[:40]):
        raw = raw[len(user_msg):].strip()

    final = polish_answer(raw) if polish else raw

    # Log session
    ses = {
        "ts": datetime.utcnow().isoformat() + "Z",
        "doc_id": DOC_ID,
        "question": question,
        "hits_meta": [
            {
                "chunk_id": h["chunk_id"],
                "node_id": h["node_id"],
                "section": h.get("section"),
                "pages": h.get("pages"),
                "score": h.get("score"),
            } for h in hits
        ],
        "contexts": ctx_texts,
        "prompt_tokens_hint": len(full_prompt.split()),
        "gen_ms": gen_ms,
        "raw_answer": raw,
        "final_answer": final,
    }
    fp = SESS_DIR / f"session_{int(time.time())}.json"
    with open(fp, "w", encoding="utf-8") as f:
        json.dump(ses, f, ensure_ascii=False, indent=2)
    print(f"[OK] session saved → {fp}")

    return {"raw": raw, "final": final, "hits": hits, "session_path": str(fp)}

# --- Smoke tests (domain-appropriate, no forced page suffix) ---
probes = [
    "Summarize the key billing rules in the Introduction and General Guidelines.",
    "What documentation is required for Evaluation and Management visits?",
    "Give 4 bullets on Physical Medicine therapy coverage limits.",
]
for q in probes:
    print("\nQ:", q)
    res = answer_with_rag(q, polish=True)
    print("Answer:\n", res["final"])


Q: Summarize the key billing rules in the Introduction and General Guidelines.


The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[OK] session saved → D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sessions\session_1756664269.json
Answer:
 <|system|>
You are a focused assistant for a single PDF (New York State Workers’ Compensation Medical Fee Schedules). Answer STRICTLY from the provided contexts. If the contexts are insufficient, say so. Prefer concise, practical wording. Include bracketed citations from the context tail exactly as given (e.g., [pp. 21–22] or [chunk <id>]). Do not invent page numbers.
Document ID: New_York_State_Workers_Compensation_Medical_Fee
When summarizing rules, prefer short bullets; when answering factual questions, give a direct answer first.
</|system|>
<|user|>
Instructions:
1) Use only the facts in the contexts.
2) Keep citations in-square-brackets as they appear at the end of the relevant sentence(s).
3) If unsure, say you don’t have enough info.

Question: Summarize the key billing rules in the Introduction and General G

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[OK] session saved → D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sessions\session_1756664319.json
Answer:
 <|system|>
You are a focused assistant for a single PDF (New York State Workers’ Compensation Medical Fee Schedules). Answer STRICTLY from the provided contexts. If the contexts are insufficient, say so. Prefer concise, practical wording. Include bracketed citations from the context tail exactly as given (e.g., [pp. 21–22] or [chunk <id>]). Do not invent page numbers.
Document ID: New_York_State_Workers_Compensation_Medical_Fee
When summarizing rules, prefer short bullets; when answering factual questions, give a direct answer first.
</|system|>
<|user|>
Instructions:
1) Use only the facts in the contexts.
2) Keep citations in-square-brackets as they appear at the end of the relevant sentence(s).
3) If unsure, say you don’t have enough info.

Question: What documentation is required for Evaluation and Management visi

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[OK] session saved → D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\sessions\session_1756664395.json
Answer:
 <|system|>
You are a focused assistant for a single PDF (New York State Workers’ Compensation Medical Fee Schedules). Answer STRICTLY from the provided contexts. If the contexts are insufficient, say so. Prefer concise, practical wording. Include bracketed citations from the context tail exactly as given (e.g., [pp. 21–22] or [chunk <id>]). Do not invent page numbers.
Document ID: New_York_State_Workers_Compensation_Medical_Fee
When summarizing rules, prefer short bullets; when answering factual questions, give a direct answer first.
</|system|>
<|user|>
Instructions:
1) Use only the facts in the contexts.
2) Keep citations in-square-brackets as they appear at the end of the relevant sentence(s).
3) If unsure, say you don’t have enough info.

Question: Give 4 bullets on Physical Medicine therapy coverage limits.

---

## **12) Multi LLM Aget**

In [69]:
from pathlib import Path
import os, json, platform, sys

CWD  = Path.cwd().resolve()
ROOT = CWD if (CWD / "src").exists() else CWD.parent
if str(ROOT) not in sys.path: sys.path.append(str(ROOT))

DOC_ID = "New_York_State_Workers_Compensation_Medical_Fee"

MERGED_DIR   = ROOT / "outputs" / "lora_hf" / f"{DOC_ID}_merged"   # <- merged model here
CHUNKS_DIR   = ROOT / "data" / "runs" / DOC_ID / "chunks"
GRAPH_DIR    = ROOT / "data" / "runs" / DOC_ID / "graph" / "graph"
RERANKER_DIR = ROOT / "outputs" / "reranker" / DOC_ID
SESS_DIR     = ROOT / "data" / "runs" / DOC_ID / "sessions"
PROMPTS_DIR  = ROOT / "configs" / "prompts" / "nys_mfs"

print("ROOT:", ROOT)
print("MERGED_DIR:", MERGED_DIR)
print("CHUNKS_DIR:", CHUNKS_DIR)
print("GRAPH_DIR:", GRAPH_DIR)
print("RERANKER_DIR:", RERANKER_DIR)
print("PROMPTS_DIR:", PROMPTS_DIR)
for p in (MERGED_DIR, CHUNKS_DIR, GRAPH_DIR, RERANKER_DIR):
    assert p.exists(), f"Missing: {p}"
SESS_DIR.mkdir(parents=True, exist_ok=True)
PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

ROOT: D:\IIT BBS\Job Resources\Business Optima\pdf-agent
MERGED_DIR: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\outputs\lora_hf\New_York_State_Workers_Compensation_Medical_Fee_merged
CHUNKS_DIR: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\chunks
GRAPH_DIR: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\New_York_State_Workers_Compensation_Medical_Fee\graph\graph
RERANKER_DIR: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\outputs\reranker\New_York_State_Workers_Compensation_Medical_Fee
PROMPTS_DIR: D:\IIT BBS\Job Resources\Business Optima\pdf-agent\configs\prompts\nys_mfs
